# 🍃 Notebook 01: Exploratory Atmospheric Data Analysis (EDA)
### Air Quality Dynamics & Seasonal Spikes in Delhi & Lucknow
**Author:** Khushi Yadav ([@khushiyadav35](https://github.com/khushiyadav35))  
**Degree:** B.Tech Computer Science & Engineering | Final Year Capstone  
**Standard:** Central Pollution Control Board (CPCB) India NAQI (2014)  

---
## 1. Problem Formulation & Regional Context
The Indo-Gangetic Plain (IGP) hosts some of the most critical air quality hotspots globally. Two focal cities represent differing atmospheric microclimates:
1. **Delhi (National Capital Region):** Megacity density, heavy vehicular ingress, brick kilns, and severe post-monsoon stubble burning in neighboring states (Punjab/Haryana).
2. **Lucknow (Uttar Pradesh):** Dense valley topography, biomass combustion, construction particulates, and severe nocturnal atmospheric boundary layer lowering.

This notebook explores multi-year hourly time-series data to identify diurnal patterns, seasonal inversions, and meteorological correlations.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.ingestion import load_dataset_for_city
from src.preprocessing.cpcb_calculator import CPCBCalculator

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

## 2. Ingesting Atmospheric Datasets & Computing CPCB NAQI

In [2]:
delhi_raw = load_dataset_for_city('Delhi')
lucknow_raw = load_dataset_for_city('Lucknow')

delhi_df = CPCBCalculator.compute_dataframe_naqi(delhi_raw)
lucknow_df = CPCBCalculator.compute_dataframe_naqi(lucknow_raw)

print(f"Delhi Records: {delhi_df.shape}")
print(f"Lucknow Records: {lucknow_df.shape}")
delhi_df.head(5)

## 3. CPCB Category Frequency Distribution
Analyzing the percentage of hours spent in each Indian NAQI tier (Good to Severe).

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
colors = ['#10B981', '#84CC16', '#EAB308', '#F97316', '#EF4444', '#7F1D1D']

for ax, (city, df) in zip(axes, [('Delhi', delhi_df), ('Lucknow', lucknow_df)]):
    counts = df['aqi_category'].value_counts(normalize=True).reindex(order).fillna(0) * 100
    sns.barplot(x=counts.index, y=counts.values, palette=colors, ax=ax, edgecolor='black', alpha=0.9)
    ax.set_title(f'{city} - NAQI Category Distribution (%)', fontweight='bold')
    ax.set_ylabel('Percentage of Observations (%)')
    ax.set_ylim(0, 45)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.8), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Diurnal (Hourly) Pollution Cycles
Observing rush hour vehicular and thermal inversion cycles across the 24 hours of the day.

In [4]:
delhi_df['hour'] = pd.to_datetime(delhi_df['timestamp']).dt.hour
lucknow_df['hour'] = pd.to_datetime(lucknow_df['timestamp']).dt.hour

d_hourly = delhi_df.groupby('hour')['cpcb_aqi'].mean()
l_hourly = lucknow_df.groupby('hour')['cpcb_aqi'].mean()

plt.figure(figsize=(11, 5))
plt.plot(d_hourly.index, d_hourly.values, marker='o', linewidth=2.5, color='#DC2626', label='Delhi (Mean NAQI)')
plt.plot(l_hourly.index, l_hourly.values, marker='s', linewidth=2.5, color='#2563EB', label='Lucknow (Mean NAQI)')
plt.axhline(200, color='orange', linestyle='--', label='Poor Threshold (200)')
plt.axhline(300, color='red', linestyle='--', label='Very Poor Threshold (300)')

plt.title('Diurnal NAQI Trajectory across 24 Hours', fontweight='bold')
plt.xlabel('Hour of Day (IST)')
plt.ylabel('Mean National Air Quality Index (NAQI)')
plt.xticks(range(0, 24))
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 5. Meteorological Drivers & Correlation Analysis
Correlation between particulate concentrations ($PM_{2.5}, PM_{10}$) and weather parameters (wind speed, temperature, relative humidity).

In [5]:
corr_cols = ['pm25', 'pm10', 'no2', 'so2', 'temperature', 'relative_humidity', 'wind_speed', 'cpcb_aqi']
corr_matrix = delhi_df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Atmospheric & Meteorological Cross-Correlation Matrix (Delhi)', fontweight='bold')
plt.tight_layout()
plt.show()